# S21 · Last-mile routing with a nearest-neighbour rule

A delivery van must visit a set of city stops and return to the depot. We build a
simple, fast "always drive to the nearest stop you have not visited yet" route,
plot it, and then see why finding the truly shortest route becomes impossible as
the number of stops grows. This is the everyday problem behind Delhivery, Flipkart
Ekart and Swiggy fleets.

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the plain note
  above each one. This notebook installs nothing.
- We measure how far apart two stops are with straight-line distance. If that
  idea is new, the picture is in `primers/distance_between_points.md`.
- New to the idea of a solver and a "good enough" rule? See
  `primers/optimization_objectives_and_constraints.md`.
- Already confident? Run the simple cells, then look for the **Stretch
  (optional)** cell near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

This notebook only uses libraries that Google Colab already has, so there is
nothing to install, on Colab or on your own machine.

In [ ]:
# Nothing to install - Colab already ships numpy, matplotlib and math.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                  # fast maths on tables of numbers
import matplotlib.pyplot as plt       # drawing the map and the route
import math                           # for the factorial, later on

## Step 1 — make some city stops

Each stop is a point on a map, given by an (x, y) coordinate. We scatter 10
random stops in a square and treat stop 0 as the **depot**, where the van starts
and must return. We set a seed so the map is the same every run.

In [ ]:
# Set a seed so the random stop locations are the same every time.
np.random.seed(0)

number_of_stops = 10

# Random (x, y) coordinates for each stop, between 0 and 1.
# stops[k] is the position of stop number k. Stop 0 is the depot.
stops = np.random.rand(number_of_stops, 2)

print("Stop coordinates (x, y):")
for stop_index in range(number_of_stops):
    print("  stop", stop_index, ":", stops[stop_index].round(2))

## Step 2 — always look at the data first

Before anything else, plot the stops and just look. The depot (stop 0) is drawn
as a big star.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(stops[:, 0], stops[:, 1], color="#2E75B6", s=80)

# Mark the depot (stop 0) specially.
plt.scatter(stops[0, 0], stops[0, 1], color="#27AE60", s=300, marker="*",
            label="depot (stop 0)")

# Label each stop with its number.
for stop_index in range(number_of_stops):
    plt.text(stops[stop_index, 0] + 0.012, stops[stop_index, 1] + 0.012,
             str(stop_index), fontsize=11)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Delivery stops on the map")
plt.legend()
plt.show()

## Step 3 — a helper to measure distance between two stops

We use ordinary straight-line (Euclidean) distance. We write one small function so
the rest of the code reads clearly.

In [ ]:
def distance_between(stop_a, stop_b):
    # stop_a and stop_b are (x, y) coordinates.
    gap_x = stop_a[0] - stop_b[0]
    gap_y = stop_a[1] - stop_b[1]
    # math.hypot gives sqrt(gap_x**2 + gap_y**2): the straight-line distance.
    return math.hypot(gap_x, gap_y)

# Quick check: distance from the depot (stop 0) to stop 1.
print("Distance from stop 0 to stop 1:",
      round(distance_between(stops[0], stops[1]), 3))

## Step 4 — the nearest-neighbour rule

A **heuristic** is a simple rule that gives a good (not necessarily perfect)
answer quickly. Ours is: start at the depot, then repeatedly drive to the
**nearest stop we have not visited yet**, until every stop is visited. Finally,
return to the depot.

In [ ]:
# We start at the depot, stop 0.
current_stop = 0

# Which stops still need visiting (everything except the depot to begin with).
not_visited = list(range(1, number_of_stops))

# Build the route as a list of stop numbers, starting at the depot.
route = [0]

# Keep going until every stop has been visited.
while len(not_visited) > 0:
    # Find the unvisited stop that is closest to where we are now.
    nearest_stop = not_visited[0]
    nearest_distance = distance_between(stops[current_stop], stops[nearest_stop])
    for candidate in not_visited:
        candidate_distance = distance_between(stops[current_stop], stops[candidate])
        if candidate_distance < nearest_distance:
            nearest_stop = candidate
            nearest_distance = candidate_distance

    # Drive there: add it to the route and mark it visited.
    route.append(nearest_stop)
    not_visited.remove(nearest_stop)
    current_stop = nearest_stop

# Finally, return to the depot to close the loop.
route.append(0)

print("Route (order of stops):", route)

## Step 5 — measure how long the route is

Add up the distance of every leg of the trip, from the depot all the way back to
the depot.

In [ ]:
total_distance = 0
for leg in range(len(route) - 1):
    from_stop = route[leg]
    to_stop = route[leg + 1]
    total_distance = total_distance + distance_between(stops[from_stop], stops[to_stop])

print("Total route distance:", round(total_distance, 3))

## Step 6 — plot the route

We draw a line through the stops in the order the rule chose. A good route has few
long jumps and few crossings.

In [ ]:
plt.figure(figsize=(7, 6))

# Draw each leg of the route as a line between consecutive stops.
for leg in range(len(route) - 1):
    from_stop = route[leg]
    to_stop = route[leg + 1]
    plt.plot([stops[from_stop, 0], stops[to_stop, 0]],
             [stops[from_stop, 1], stops[to_stop, 1]],
             color="#C0392B", linewidth=2, zorder=1)

# Draw the stops on top.
plt.scatter(stops[:, 0], stops[:, 1], color="#2E75B6", s=80, zorder=2)
plt.scatter(stops[0, 0], stops[0, 1], color="#27AE60", s=300, marker="*",
            zorder=3, label="depot")

for stop_index in range(number_of_stops):
    plt.text(stops[stop_index, 0] + 0.012, stops[stop_index, 1] + 0.012,
             str(stop_index), fontsize=11)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Nearest-neighbour delivery route (distance = "
          + str(round(total_distance, 3)) + ")")
plt.legend()
plt.show()

## Step 7 — why not just find the *perfect* route?

Finding the genuinely shortest route that visits every stop once is the famous
**Travelling Salesman Problem (TSP)**. The only sure way is, in effect, to check
the possible orderings of the stops, and the number of orderings grows as a
**factorial**. Let's count how fast that blows up.

In [ ]:
print("stops | number of possible routes")
print("------+---------------------------")
for n in range(5, 16):
    # With n stops there are roughly (n-1)! / 2 distinct round-trip routes.
    number_of_routes = math.factorial(n - 1) // 2
    print(" {:>4} | {:,}".format(n, number_of_routes))

## Step 8 — the explosion, in a picture

On a log scale (each step up the axis means ten times as many), the count shoots
up almost vertically. Even a fast computer checking millions of routes a second
cannot keep up once you pass a few dozen stops, and a real metro has thousands.
This is a **combinatorial explosion**.

In [ ]:
stop_counts = list(range(5, 21))
route_counts = []
for n in stop_counts:
    route_counts.append(math.factorial(n - 1) // 2)

plt.figure(figsize=(7, 5))
plt.plot(stop_counts, route_counts, color="#C0392B", marker="o")
plt.yscale("log")          # log scale because the numbers grow so fast
plt.xlabel("number of stops")
plt.ylabel("number of possible routes (log scale)")
plt.title("Exact TSP explodes combinatorially")
plt.show()

### Stretch (optional) — how close did the fast rule get to perfect?

Skip this if you are new to code. For a *small* set of stops we can still find the
true shortest route by brute force and see how good our quick rule was. We shrink
to 8 stops, because 7! = 5,040 routes is checkable in a blink, while 12 stops
would already be millions. That shrinking is itself the lesson: exact only works
when the problem is tiny.

In [ ]:
import itertools

# A smaller instance: the first 8 stops, depot still stop 0.
small_stops = stops[:8]
small_n = len(small_stops)

def route_length(order):
    # order is a full trip: a list of stop indices starting and ending at 0.
    total = 0.0
    for leg in range(len(order) - 1):
        total += distance_between(small_stops[order[leg]], small_stops[order[leg + 1]])
    return total

# Nearest-neighbour on the small instance.
here = 0
todo = list(range(1, small_n))
nn_order = [0]
while todo:
    nxt = min(todo, key=lambda s: distance_between(small_stops[here], small_stops[s]))
    nn_order.append(nxt)
    todo.remove(nxt)
    here = nxt
nn_order.append(0)
nn_length = route_length(nn_order)

# True shortest route by trying every ordering of stops 1..7 (depot fixed at 0).
best_length = float("inf")
for middle in itertools.permutations(range(1, small_n)):
    candidate = [0] + list(middle) + [0]
    length = route_length(candidate)
    if length < best_length:
        best_length = length

print("Nearest-neighbour distance:", round(nn_length, 3))
print("True shortest distance    :", round(best_length, 3))
gap_percent = 100 * (nn_length - best_length) / best_length
print("The quick rule was within", round(gap_percent, 1),
      "% of perfect - fast and pretty good.")

## What you just did

You built a complete delivery route with a simple nearest-neighbour rule (start at
the depot, always go to the closest unvisited stop) and plotted it. Then you saw
why nobody solves real routing exactly: the number of possible routes grows like a
factorial, so it is hopeless past a few dozen stops. This is why Delhivery,
Flipkart and Swiggy route their fleets with fast heuristics under a time budget,
accepting a very good answer instead of waiting forever for a perfect one.

**Try it yourself:** start the route from a different stop, or add more stops, and
watch how the total distance changes.